In [74]:
%pip install tabulate

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
import pickle
from tabulate import tabulate


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [105]:
################ Load model results
# Demand model results
with open('demand_model_results.pkl', 'rb') as f:
     demand_param = pickle.load(f)

# Charging station model results
with open('charging_station_model_results.pkl', 'rb') as f:
     charging_param = pickle.load(f)

# Import data
df = pd.read_csv('demand_counterfactual.csv')
supply = pd.read_csv('supply_counterfactual.csv')

# Calculate number of new stations using charging_stations_stock_lag directly
df['new_stations'] = df['charging_stations_stock'] - df['charging_stations_stock_lag']

# Add EV_stock from supply to df if columns exist
if 'year' in df.columns and 'province' in df.columns:
    df = df.merge(supply[['year', 'province', 'EV_stock']], on=['year', 'province'], how='left')
else:
    print("Either 'year' or 'province' column is missing in df. Please check your data.")

# Ensure required columns are present in df by merging from supply if missing
required_cols = ['sub_fix', 'sub_ope', 'time_trend', 'num_models_in_market', 'sales_weighted_avg_range']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
	df = df.merge(
		supply[['year', 'province'] + missing_cols],
		on=['year', 'province'],
		how='left'
	)

In [108]:
# Check fuel types in the data
print("Unique fuel types:", df['fuel_type'].unique())
print("Fuel type counts:")
print(df['fuel_type'].value_counts())

# Check some samples to understand the data structure
print("\nSample data:")
print(df[['fuel_type', 'sub', 'nat', 'sub_PHEV', 'sub_BEV_1', 'sub_BEV_2', 'sub_BEV_3']].head(10))

Unique fuel types: ['BEV' 'PHEV' 'Gas']
Fuel type counts:
fuel_type
Gas     18833
BEV      8636
PHEV     6076
Name: count, dtype: int64

Sample data:
  fuel_type  sub  nat  sub_PHEV  sub_BEV_1  sub_BEV_2  sub_BEV_3
0       BEV  0.0  0.0       1.0        1.8        1.8        2.5
1       BEV  0.0  0.0       1.0        1.8        1.8        2.5
2       BEV  0.0  0.0       1.0        1.8        1.8        2.5
3      PHEV  0.0  0.0       1.0        1.8        1.8        2.5
4       BEV  0.0  0.0       1.0        1.8        1.8        2.5
5      PHEV  0.0  0.0       1.0        1.8        1.8        2.5
6       BEV  0.0  0.0       1.0        1.8        1.8        2.5
7      PHEV  0.0  0.0       1.0        1.8        1.8        2.5
8      PHEV  0.0  0.0       1.0        1.8        1.8        2.5
9      PHEV  0.0  0.0       1.0        1.8        1.8        2.5


In [110]:
def create_ev_only_subsidy_scenario(df, sub=0, nat=0, PHEV=0, BEV1=0, BEV2=0, BEV3=0, t=1, gas_tax=None):
    """
    Create a counterfactual scenario where only EVs (BEV and PHEV) receive subsidies.
    Gas vehicles get all subsidies set to 0.
    
    Parameters:
    - df: DataFrame with the data
    - sub, nat, PHEV, BEV1, BEV2, BEV3, t: subsidy values for EVs
    - gas_tax: Tax rate for Gas vehicles (if None, uses original tax rate from data)
    
    Returns:
    - List of dicts for each observation, with subsidies set appropriately
    """
    counterfactual_scenario = []
    
    for _, row in df.iterrows():
        if row['fuel_type'] == 'Gas':
            # Gas vehicles get no subsidies
            scenario_dict = {
                'sub': 0,
                'nat': 0,
                'PHEV': 0,
                'BEV1': 0,
                'BEV2': 0,
                'BEV3': 0,
                't': gas_tax if gas_tax is not None else row['Tax'],  # Keep original tax rate or use specified
            }
        else:
            # EVs (BEV and PHEV) get the specified subsidies
            scenario_dict = {
                'sub': sub,
                'nat': nat,
                'PHEV': PHEV,
                'BEV1': BEV1,
                'BEV2': BEV2,
                'BEV3': BEV3,
                't': t,
            }
        counterfactual_scenario.append(scenario_dict)
    
    return counterfactual_scenario

print("Updated function to create EV-only subsidy scenarios defined successfully!")

Updated function to create EV-only subsidy scenarios defined successfully!


In [106]:
# Nested net price function
def compute_net_price(share, manual_subsidy=None):
        rg = share['range']
        p = share['prices']
        if manual_subsidy is not None:
            sub = manual_subsidy.get('sub', share['sub'])
            nat = manual_subsidy.get('nat', share['nat'])
            PHEV = manual_subsidy.get('PHEV', share['sub_PHEV']) if share['fuel_type'] == 'PHEV' else 0
            BEV1 = manual_subsidy.get('BEV1', share['sub_BEV_1']) if share['fuel_type'] == 'BEV' else 0
            BEV2 = manual_subsidy.get('BEV2', share['sub_BEV_2']) if share['fuel_type'] == 'BEV' else 0
            BEV3 = manual_subsidy.get('BEV3', share['sub_BEV_3']) if share['fuel_type'] == 'BEV' else 0
            t = manual_subsidy.get('t', share['Tax'])
        else:
            sub = share['sub']
            nat = share['nat']
            PHEV = share['sub_PHEV'] if share['fuel_type'] == 'PHEV' else 0
            BEV1 = share['sub_BEV_1'] if share['fuel_type'] == 'BEV' else 0
            BEV2 = share['sub_BEV_2'] if share['fuel_type'] == 'BEV' else 0
            BEV3 = share['sub_BEV_3'] if share['fuel_type'] == 'BEV' else 0
            t = share['Tax']
        def range_category(r, BEV1, BEV2, BEV3):
            if 250 <= r < 300:
                return BEV1
            elif 300 <= r < 400:
                return BEV2
            elif 400 <= r:
                return BEV3
            else:
                return 0
        BEV = range_category(rg, BEV1, BEV2, BEV3)

        if nat == 0:
            return p * t - sub - PHEV - BEV
        elif nat == 1:
            return p * t - nat * 10000 * sub * (BEV + PHEV) - PHEV - BEV
        else:
            return 0

In [77]:
def compute_delta_jm(X_jm, demand_par, purchase_subsidy=None, counterfactual_subsidy=None):
    """
    Compute mean utilities δ_jm for nested logit model:
    δ_jm = β_price * net_prices + β_range * range + β_power * power + β_battery * battery_capacity 
           + β_EV * is_electric + β_N * log_charging_stock_hat 
           + β_N_EV * (is_electric × log_charging_stock_hat) + ξ_jm

    If purchase_subsidy is provided (array-like, same length as X_jm), it will be subtracted from net_prices
    to allow counterfactuals with different purchase subsidy levels.

    If counterfactual_subsidy is provided (list of dicts, same length as X_jm), it will be used to override the subsidy
    for each observation (e.g., set to 0 for no subsidy scenario).
    """
    # Copy to avoid modifying original DataFrame
    X = X_jm.copy()

    # If counterfactual_subsidy is provided, use it to override subsidy in net price calculation
    if counterfactual_subsidy is not None:
        X['net_prices'] = [
            compute_net_price(row, manual_subsidy=counterfactual_subsidy[i])
            for i, row in X.iterrows()
        ]
        if purchase_subsidy is not None:
            X['net_prices'] = X['net_prices'] - purchase_subsidy
    elif purchase_subsidy is not None:
        X['net_prices'] = X['net_prices'] - purchase_subsidy

    # Use log_charging_stock_hat instead of log_charging_stock to match demand model
    X['log_charging_stock_hat'] = X['log_charging_stock']
    
    # Compute utility following the exact demand model specification
    delta_jm = (
        demand_par.params['net_prices'] * X['net_prices'] +
        demand_par.params['range'] * X['range'] +
        demand_par.params['power'] * X['power'] +
        demand_par.params['battery_capacity'] * X['battery_capacity'] +
        demand_par.params['is_electric'] * X['is_electric'] +
        demand_par.params['log_charging_stock_hat'] * X['log_charging_stock_hat'] +
        demand_par.params['is_electric:log_charging_stock_hat'] * (X['is_electric'] * X['log_charging_stock_hat']) +
        demand_par.resids
    )
    
    return delta_jm


In [78]:
################ Nested logit shares
def nested_logit_shares(delta_jm, rho, group_membership, market_ids=None):
    """
    Compute nested logit predicted shares for every product in every market,
    using mean utilities of all product-market pairs.

    Parameters
    ----------
    delta_jm : array-like
        Mean utilities for all product-market pairs (length = number of rows in df)
    rho : float
        Nesting parameter (0 ≤ ρ < 1)
    group_membership : array-like
        Nest assignment for each product-market pair (same length as delta_jm)
    market_ids : array-like or None
        Market assignment for each product-market pair (same length as delta_jm).
        If None, treats all rows as one market.

    Returns
    -------
    s_j : np.ndarray
        Share for each product-market pair (same order as input)
    s_0 : np.ndarray
        Outside good share for each market (same order as unique market_ids)
    """
    delta_jm = np.array(delta_jm)
    group_membership = np.array(group_membership)
    if market_ids is None:
        market_ids = np.zeros(len(delta_jm), dtype=int)
    else:
        market_ids = np.array(market_ids)

    s_j = np.zeros_like(delta_jm)
    unique_markets = np.unique(market_ids)
    s_0 = np.zeros(len(unique_markets))

    for i, m in enumerate(unique_markets):
        mask_market = (market_ids == m)
        delta_m = delta_jm[mask_market]
        group_m = group_membership[mask_market]
        unique_groups = np.unique(group_m)
        D_g = {}
        for g in unique_groups:
            mask_g = (group_m == g)
            D_g[g] = np.sum(np.exp(delta_m[mask_g] / (1 - rho)))
        log_ratio = np.zeros_like(delta_m)
        for j, (delta, g) in enumerate(zip(delta_m, group_m)):
            log_ratio[j] = delta / (1 - rho) - rho * np.log(D_g[g])
        sum_exp = np.sum(np.exp(log_ratio))
        s_0[i] = 1 / (1 + sum_exp)
        s_j[mask_market] = np.exp(log_ratio) * s_0[i]

    return s_j, s_0

In [79]:
################ Function to compute total spendings on subsidies
# Calculate number of new stations as the difference between charging stations stock and its lag for each province
def calculate_total_gov_spending(df, sales, net_prices, purchase_subsidy, sub_fix, sub_ope, N_market, new_stations):
    """
    Calculate total government spending on subsidies:
    - Demand subsidies: difference between prices and net_prices for each sale
    - Purchase subsidies: purchase_subsidy * sales (if provided)
    - Charging subsidies: avg_charging * N_market + sub_fix * new_stations
    """
    # Product-level spending (vehicle sales subsidies)
    purchase_subsidy = purchase_subsidy if purchase_subsidy is not None else 0
    gov_spending_product = sales * (df['prices'] - np.array(net_prices)) + purchase_subsidy * sales

    # Market-level spending (charging station subsidies)
    avg_charging = 24 * 0.13 * 60 * sub_ope  # Operating subsidy per charging stick
    gov_spending_market = avg_charging * N_market + sub_fix * new_stations

    total_spending = np.sum(gov_spending_product) * 10000 + np.sum(gov_spending_market)


    return total_spending

In [85]:
################ Fixed-point iteration for EV-station equilibrium
def fixed_point_iteration(
    design_matrix_init,
    demand_par,
    charging_par,
    rho,
    group_membership,
    max_iter=100,
    tol=1e-6,
    demand_subsidy=None,    # 1. Counterfactual demand-side subsidy (list of dicts for compute_delta_jm)
    purchase_subsidy=None,  # 2. Counterfactual purchase subsidy (array for compute_delta_jm)
    supply_subsidy=None     # 3. Counterfactual supply-side subsidy (mask for charging stations)
):
    """
    Fixed-point iteration for EV-station equilibrium at the market level.
    demand_subsidy: list of dicts, one per row, passed to compute_delta_jm as counterfactual_subsidy.
    purchase_subsidy: array-like, same length as df, passed to compute_delta_jm as purchase_subsidy.
    supply_subsidy: dict, DataFrame, array, or list, specifying which markets have supply-side subsidies.
    """
    df = design_matrix_init.copy()
    market_ids = df['market_ids'].values
    unique_markets = np.unique(market_ids)

    # Initialize N and Q_ev for each market (take first occurrence in each market)
    N_market = df.groupby('market_ids')['charging_stations_stock'].first().reindex(unique_markets).values
    Q_ev_market = df.groupby('market_ids')['EV_stock'].first().reindex(unique_markets).values
    Q_ev_stock = df.groupby('market_ids')['EV_stock'].first().reindex(unique_markets).values

    history = {'N': [], 'Q_ev': [], 'delta': []}

    for iteration in range(max_iter):
        # Assign current market-level N and Q_ev to all rows
        market_idx_map = {m: i for i, m in enumerate(unique_markets)}
        df['N_market'] = [N_market[market_idx_map[m]] for m in market_ids]
        df['Q_ev_market'] = [Q_ev_market[market_idx_map[m]] for m in market_ids]
        df['charging_stations_stock'] = df['N_market']
        df['EV_stock'] = df['Q_ev_market']
        # Remove +1 to match original model specification
        df['log_charging_stock'] = np.log(df['charging_stations_stock'])

        # Compute mean utilities with counterfactual demand and purchase subsidies
        delta = compute_delta_jm(
            df,
            demand_par,
            purchase_subsidy=purchase_subsidy,
            counterfactual_subsidy=demand_subsidy
        )

        # Compute shares for each product-market
        s_j, _ = nested_logit_shares(delta, rho, group_membership, market_ids=market_ids)

        # Update EV stock for each market by summing EV sales in that market
        ev_mask = df['is_electric'].values == 1
        market_size = df['market_size'].values if 'market_size' in df.columns else np.ones_like(s_j)
        sales = s_j * market_size
        ev_sales = sales * ev_mask
        df['ev_sales'] = ev_sales
        Q_ev_market_new = df.groupby('market_ids')['ev_sales'].sum().reindex(unique_markets).values + Q_ev_stock

        # Update station stock for each market using the supply model
        supply_covs = df.drop_duplicates('market_ids').set_index('market_ids').reindex(unique_markets)
        if supply_subsidy is not None:
            # Ensure supply_subsidy is properly broadcast to match unique_markets length
            if isinstance(supply_subsidy, (list, tuple)) and len(supply_subsidy) == 2:
                sub_fix = np.full(len(unique_markets), supply_subsidy[0])
                sub_ope = np.full(len(unique_markets), supply_subsidy[1])
            else:
                sub_fix = supply_subsidy[0]
                sub_ope = supply_subsidy[1]
        else:
            sub_fix = supply_covs['sub_fix'].values.copy()
            sub_ope = supply_covs['sub_ope'].values.copy()

        # Initialize log_N with base terms - remove +1 to match original model
        log_N = (
            charging_par.params['log(EV_stock)'] * np.log(Q_ev_market_new) +
            charging_par.params['sub_fix'] * sub_fix +
            charging_par.params['sub_ope'] * sub_ope +
            charging_par.params['time_trend'] * supply_covs['time_trend'].values
        )
        
        # Add residuals - ensure shape compatibility
        if hasattr(charging_par, 'resids') and charging_par.resids is not None:
            resids = charging_par.resids
            if len(resids) == len(unique_markets):
                log_N += resids
            else:
                # If residuals don't match markets, use appropriate subset or skip
                print(f"Warning: Residuals shape {len(resids)} doesn't match markets {len(unique_markets)}")
                # Try to use first len(unique_markets) residuals if available
                if len(resids) >= len(unique_markets):
                    log_N += resids[:len(unique_markets)]
        
        # Add province fixed effects with improved parsing
        for k in charging_par.params.keys():
            if k.startswith('C(province)'):
                # Handle both [T.province] and [province] formats
                if '[T.' in k:
                    province = k.split('[T.')[1].rstrip(']')
                elif '[' in k and ']' in k:
                    province = k.split('[')[1].rstrip(']')
                else:
                    continue  # Skip if can't parse
                
                # Create boolean mask and ensure proper broadcasting
                mask = (supply_covs['province'] == province).astype(float).values
                if len(mask) == len(unique_markets):
                    log_N += charging_par.params[k] * mask
        
        # Remove year fixed effects as they're not in the original supply model specification
        # The original model only includes time_trend, not year fixed effects

        N_market_new = np.exp(log_N)

        # Check convergence
        delta_N = np.max(np.abs(N_market_new - N_market))
        delta_Q = np.max(np.abs(Q_ev_market_new - Q_ev_market))
        history['N'].append(N_market_new.copy())
        history['Q_ev'].append(Q_ev_market_new.copy())
        history['delta'].append(max(delta_N, delta_Q))

        if delta_N < tol and delta_Q < tol:
            print(f"Converged after {iteration+1} iterations")
            break

        # Update for next iteration
        N_market = N_market_new
        Q_ev_market = Q_ev_market_new
        
        # Fix new_station calculation - ensure proper alignment
        charging_stock_lag = df.groupby('market_ids')['charging_stations_stock_lag'].first().reindex(unique_markets).values
        new_station = N_market - charging_stock_lag
        
        if demand_subsidy is not None:
            net_prices = [
                compute_net_price(row, manual_subsidy=demand_subsidy[i])
                for i, (idx, row) in enumerate(df.iterrows())
            ]
        else:
            net_prices = df['net_prices'].values

        # Call compute_total_gov_spending to calculate total government spending
        Gov_spending = calculate_total_gov_spending(
            df,
            sales,
            net_prices,
            purchase_subsidy,
            sub_fix,
            sub_ope,
            N_market,
            new_station
        )
        
    else:
        print(f"Warning: Max iterations ({max_iter}) reached")

    return {
        'N': N_market,
        'Q_ev': Q_ev_market,
        'shares': s_j,
        'history': pd.DataFrame(history),
        'Gov_spending': Gov_spending,
    }

In [86]:
################ Compute status quo from fixed-point iteration
result = fixed_point_iteration(
	df,
	demand_param,
	charging_param,
	demand_param.params['log_sj_g'],
	df['nesting_ids'].values
)

Converged after 13 iterations


In [111]:
################ Compute counterfactual scenarios
# 0. No subsidies: set all subsidies to 0 for EVs, Gas vehicles already have 0 subsidies
demand_counterfactual_0 = create_ev_only_subsidy_scenario(
    df, 
    sub=0, nat=0, 
    PHEV=0, BEV1=0, BEV2=0, BEV3=0, 
    t=1,
    gas_tax=None  # Gas vehicles keep their original tax rate
)

supply_subsidy_0 = [0,0]

result_0 = fixed_point_iteration(
    df,
    demand_param,
    charging_param,
    demand_param.params['log_sj_g'],
    df['nesting_ids'].values,
    max_iter=100,
    tol=1e-6,
    demand_subsidy=demand_counterfactual_0,    # 1. Counterfactual demand-side subsidy (list of dicts for compute_delta_jm)
    purchase_subsidy=None,  # 2. Counterfactual purchase subsidy (array for compute_delta_jm)
    supply_subsidy=supply_subsidy_0     # 3. Counterfactual supply-side subsidy (mask for charging stations)
)

Converged after 13 iterations


In [ ]:
# Verification: Check that subsidies are correctly assigned only to EVs
print("Verification of EV-only subsidy assignment:")
print("Status quo (should have original subsidies for EVs, none for Gas):")
sample_indices = [0, 1, 2, 15000, 25000, 30000]  # Sample different fuel types
for i in sample_indices:
    if i < len(df):
        fuel_type = df.iloc[i]['fuel_type']
        original_sub = df.iloc[i]['sub']
        original_phev = df.iloc[i]['sub_PHEV'] 
        original_bev1 = df.iloc[i]['sub_BEV_1']
        print(f"Index {i}: {fuel_type} - Original: sub={original_sub}, PHEV={original_phev}, BEV1={original_bev1}")

print("\nCounterfactual 0 (should have 0 subsidies for everyone):")
for i in sample_indices:
    if i < len(demand_counterfactual_0):
        fuel_type = df.iloc[i]['fuel_type']
        cf_sub = demand_counterfactual_0[i]['sub']
        cf_phev = demand_counterfactual_0[i]['PHEV']
        cf_bev1 = demand_counterfactual_0[i]['BEV1']
        print(f"Index {i}: {fuel_type} - CF0: sub={cf_sub}, PHEV={cf_phev}, BEV1={cf_bev1}")

print("\nCounterfactual 1 (should have EV subsidies for EVs, 0 for Gas):")
for i in sample_indices:
    if i < len(demand_counterfactual_1):
        fuel_type = df.iloc[i]['fuel_type']
        cf_sub = demand_counterfactual_1[i]['sub']
        cf_phev = demand_counterfactual_1[i]['PHEV']
        cf_bev1 = demand_counterfactual_1[i]['BEV1']
        print(f"Index {i}: {fuel_type} - CF1: sub={cf_sub}, PHEV={cf_phev}, BEV1={cf_bev1}")

In [ ]:
# 1. Only demand_subsidies - EV subsidies only
demand_counterfactual_1 = create_ev_only_subsidy_scenario(
    df, 
    sub=0, nat=0, 
    PHEV=1.0, BEV1=1.8, BEV2=1.8, BEV3=2.5, 
    t=1
)

supply_subsidy_1 = [0,0]

result_1 = fixed_point_iteration(
    df,
    demand_param,
    charging_param,
    demand_param.params['log_sj_g'],
    df['nesting_ids'].values,
    max_iter=100,
    tol=1e-6,
    demand_subsidy=demand_counterfactual_1,    # 1. Counterfactual demand-side subsidy (list of dicts for compute_delta_jm)
    purchase_subsidy=None,  # 2. Counterfactual purchase subsidy (array for compute_delta_jm)
    supply_subsidy=supply_subsidy_1     # 3. Counterfactual supply-side subsidy (mask for charging stations)
)

Converged after 13 iterations


In [ ]:
# 2. Only station subsidies - no EV subsidies for anyone
demand_counterfactual_2 = create_ev_only_subsidy_scenario(
    df, 
    sub=0, nat=0, 
    PHEV=0, BEV1=0, BEV2=0, BEV3=0, 
    t=1
)

supply_subsidy_2 = None

result_2 = fixed_point_iteration(
    df,
    demand_param,
    charging_param,
    demand_param.params['log_sj_g'],
    df['nesting_ids'].values,
    max_iter=100,
    tol=1e-6,
    demand_subsidy=demand_counterfactual_2,    # 1. Counterfactual demand-side subsidy (list of dicts for compute_delta_jm)
    purchase_subsidy=None,  # 2. Counterfactual purchase subsidy (array for compute_delta_jm)
    supply_subsidy=supply_subsidy_2     # 3. Counterfactual supply-side subsidy (mask for charging stations)
)

Converged after 13 iterations


In [ ]:
# 3. Added purchase subsidies and both subsidies on - EV subsidies only
demand_counterfactual_3 = create_ev_only_subsidy_scenario(
    df, 
    sub=0, nat=0, 
    PHEV=1.8, BEV1=1.8, BEV2=1.8, BEV3=2.5, 
    t=1
)

supply_subsidy_3 = [0,0]

purchase_subsidy_3 = None

result_3 = fixed_point_iteration(
    df,
    demand_param,
    charging_param,
    demand_param.params['log_sj_g'],
    df['nesting_ids'].values,
    max_iter=100,
    tol=1e-6,
    demand_subsidy=demand_counterfactual_3,    # 1. Counterfactual demand-side subsidy (list of dicts for compute_delta_jm)
    purchase_subsidy=purchase_subsidy_3,  # 2. Counterfactual purchase subsidy (array for compute_delta_jm)
    supply_subsidy=supply_subsidy_3     # 3. Counterfactual supply-side subsidy (mask for charging stations)
)

Converged after 14 iterations


In [ ]:
# 4. Comparison between tax and purchase subsidies - EV subsidies only
demand_counterfactual_4 = create_ev_only_subsidy_scenario(
    df, 
    sub=0, nat=0, 
    PHEV=0, BEV1=0, BEV2=0, BEV3=0, 
    t=0.911,  # Tax reduction only for EVs
    gas_tax=None  # Gas vehicles keep their original tax rate
)

supply_subsidy_4 = [0,0]

purchase_subsidy_4 = None

result_4 = fixed_point_iteration(
    df,
    demand_param,
    charging_param,
    demand_param.params['log_sj_g'],
    df['nesting_ids'].values,
    max_iter=100,
    tol=1e-6,
    demand_subsidy=demand_counterfactual_4,    # 1. Counterfactual demand-side subsidy (list of dicts for compute_delta_jm)
    purchase_subsidy=purchase_subsidy_4,  # 2. Counterfactual purchase subsidy (array for compute_delta_jm)
    supply_subsidy=supply_subsidy_4     # 3. Counterfactual supply-side subsidy (mask for charging stations)
)

Converged after 13 iterations


In [ ]:
# 5. Purchase subsidies - EV subsidies only
demand_counterfactual_5 = create_ev_only_subsidy_scenario(
    df, 
    sub=0, nat=0, 
    PHEV=0, BEV1=0, BEV2=0, BEV3=0, 
    t=1,
    gas_tax=None  # Gas vehicles keep their original tax rate
)

supply_subsidy_5 = [0,0]

# Create purchase subsidy array: only EVs get the subsidy
purchase_subsidy_5 = np.array([
    2 if row['fuel_type'] in ['BEV', 'PHEV'] else 0 
    for _, row in df.iterrows()
])

result_5 = fixed_point_iteration(
    df,
    demand_param,
    charging_param,
    demand_param.params['log_sj_g'],
    df['nesting_ids'].values,
    max_iter=100,
    tol=1e-6,
    demand_subsidy=demand_counterfactual_5,    # 1. Counterfactual demand-side subsidy (list of dicts for compute_delta_jm)
    purchase_subsidy=purchase_subsidy_5,  # 2. Counterfactual purchase subsidy (array for compute_delta_jm)
    supply_subsidy=supply_subsidy_5     # 3. Counterfactual supply-side subsidy (mask for charging stations)
)

Converged after 14 iterations


In [104]:
# Compute total EV sales and total N for all markets in 2023, display only these two measures and government spending in relative terms to no-incentive case

def get_totals(result, df, year=2023):
    # Find market_ids for 2023
    if 'year' in df.columns and 'market_ids' in df.columns:
        market_ids_2023 = df.loc[df['year'] == year, 'market_ids'].unique()
        idx_2023 = [i for i, m in enumerate(np.unique(df['market_ids'])) if m in market_ids_2023]
    else:
        idx_2023 = slice(None)  # fallback: use all markets

    total_ev_sales = np.sum(result['Q_ev'])
    total_N_2023 = np.sum(result['N'][idx_2023])
    gov_spending = result['Gov_spending']
    return total_ev_sales, total_N_2023, gov_spending

# Get totals for each scenario
ev_sales_0, N_2023_0, gov_0 = get_totals(result_0, df)
ev_sales_1, N_2023_1, gov_1 = get_totals(result_1, df)
ev_sales_2, N_2023_2, gov_2 = get_totals(result_2, df)
ev_sales_3, N_2023_3, gov_3 = get_totals(result_3, df)
ev_sales_4, N_2023_4, gov_4 = get_totals(result_4, df)
ev_sales_5, N_2023_5, gov_5 = get_totals(result_5, df)

# Calculate efficiency metrics (per unit of government spending)
def calc_efficiency(delta_ev, delta_piles, gov_spending):
    if gov_spending != 0:
        ev_per_spending = delta_ev / gov_spending * 1e6  # Per million yuan
        piles_per_spending = delta_piles / gov_spending * 1e6  # Per million yuan
        return ev_per_spending, piles_per_spending
    else:
        return 0, 0

# Calculate delta values and efficiency metrics for each scenario
scenarios_data = []
for i, (ev_sales, N_2023, gov) in enumerate([
    (ev_sales_1, N_2023_1, gov_1), (ev_sales_2, N_2023_2, gov_2), 
    (ev_sales_3, N_2023_3, gov_3), (ev_sales_4, N_2023_4, gov_4), 
    (ev_sales_5, N_2023_5, gov_5)
]):
    delta_ev = ev_sales - ev_sales_0
    delta_piles = N_2023 - N_2023_0
    delta_gov = gov - gov_0
    ev_eff, piles_eff = calc_efficiency(delta_ev, delta_piles, delta_gov)
    scenarios_data.append((delta_ev, delta_piles, delta_gov, ev_eff, piles_eff))

# Create comprehensive table with all scenarios
summary_data = []

# Add main scenario rows
scenario_names = ['No Subsidies', 'Demand Only', 'Station Only', 'Purchase Incentive', 'Tax Reduction', 'Purchase Subsidy']
for i, name in enumerate(scenario_names):
    if i == 0:  # No subsidies baseline
        summary_data.append([
            name,
            "0 (0%)",
            "0 (0%)", 
            "0",
            "-",
            "-"
        ])
    else:
        idx = i - 1  # Adjust for baseline
        delta_ev_pct = f"{scenarios_data[idx][0]:.0f} ({100 * scenarios_data[idx][0] / ev_sales_0:.1f}%)" if ev_sales_0 != 0 else f"{scenarios_data[idx][0]:.0f} (0%)"
        delta_piles_pct = f"{scenarios_data[idx][1]:.0f} ({100 * scenarios_data[idx][1] / N_2023_0:.1f}%)" if N_2023_0 != 0 else f"{scenarios_data[idx][1]:.0f} (0%)"
        gov_spending = f"{scenarios_data[idx][2]:.0f}"
        ev_efficiency = f"{scenarios_data[idx][3]:.2f}"
        piles_efficiency = f"{scenarios_data[idx][4]:.2f}"
        
        summary_data.append([
            name,
            delta_ev_pct,
            delta_piles_pct,
            gov_spending,
            ev_efficiency,
            piles_efficiency
        ])

# Create DataFrame
summary_df = pd.DataFrame(summary_data, columns=[
    'Scenario', 
    '$\\Delta$ EV', 
    '$\\Delta$ Piles', 
    'Gov Spending (¥)',
    '$\\Delta$ EV per M¥',
    '$\\Delta$ Piles per M¥'
])

latex_table = tabulate(
    summary_df,
    headers='keys',
    tablefmt='latex',
    showindex=False
)

print(latex_table)

\begin{tabular}{lllrll}
\hline
 Scenario           & \$\textbackslash{}Delta\$ EV    & \$\textbackslash{}Delta\$ Piles   &   Gov Spending (¥) & \$\textbackslash{}Delta\$ EV per M¥   & \$\textbackslash{}Delta\$ Piles per M¥   \\
\hline
 No Subsidies       & 0 (0\%)         & 0 (0\%)           &                  0 & -                    & -                       \\
 Demand Only        & 1502442 (3.0\%) & 49538 (2.1\%)     &       556814474623 & 2.70                 & 0.09                    \\
 Station Only       & 26329 (0.1\%)   & 20086 (0.8\%)     &          131276814 & 200.56               & 153.00                  \\
 Purchase Incentive & 1086160 (2.2\%) & 43635 (1.8\%)     &       106767394151 & 10.17                & 0.41                    \\
 Tax Reduction      & 1057867 (2.1\%) & 44429 (1.8\%)     &       490763192571 & 2.16                 & 0.09                    \\
 Purchase Subsidy   & 1167354 (2.3\%) & 43403 (1.8\%)     &       567738231524 & 2.06                 & 0.08  